# Unit 4 — Multi-turn conversations

Agents are **stateless**. Sessions are **stateful**. Pass `session=session` to every `agent.run(...)` call and the agent remembers.

In [ ]:
from dotenv import load_dotenv
load_dotenv(dotenv_path="../.env")

from typing import Annotated
from pydantic import Field
from agent_framework.openai import OpenAIChatCompletionClient

def get_weather(city: Annotated[str, Field(description="City name.")]) -> str:
    """Return the current weather for a city."""
    data = {"berlin": "12°C, overcast", "munich": "10°C, light rain", "paris": "14°C, clear"}
    return data.get(city.lower(), f"{city}: unavailable")

agent = OpenAIChatCompletionClient().as_agent(
    name="Chatty",
    instructions="You are concise. Use the weather tool when relevant. Refer back to earlier turns when it helps.",
    tools=[get_weather],
)

## Demo A: without a session — it forgets

Two separate `.run()` calls. No session passed. The second call has no idea what the first one was about.

In [ ]:
r1 = await agent.run("What's the weather in Berlin?")
print("1:", r1.text)
r2 = await agent.run("What was the city I just asked about?")
print("2:", r2.text)

## Demo B: with a session — it remembers

`agent.create_session()` creates an in-memory conversation. Pass it to every `.run()` and the transcript accumulates.

In [ ]:
session = agent.create_session()

r1 = await agent.run("What's the weather in Berlin?", session=session)
print("1:", r1.text)

r2 = await agent.run("And in Munich?", session=session)
print("2:", r2.text)

r3 = await agent.run("Which of those is colder?", session=session)
print("3:", r3.text)

r4 = await agent.run("What was the very first city I mentioned?", session=session)
print("4:", r4.text)

## Caveat: sessions grow forever

Every turn appends to the session. After 100 turns you'll hit context limits. In production you'd summarize older turns; for ~20 it's fine.